In [1]:
import time
import pandas as pd
from pyspark.sql import SparkSession, functions as F, Window

spark = SparkSession.builder.appName("optimisation").master("local[*]").getOrCreate()
GOLD = "/home/work/data/gold/"

foi = spark.read.parquet(GOLD + "fact_order_items")
fo = spark.read.parquet(GOLD + "fact_orders")
dp = spark.read.parquet(GOLD + "dim_product")
ds = spark.read.parquet(GOLD + "dim_seller")

for k in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled",
          "spark.sql.adaptive.coalescePartitions.enabled", "spark.sql.autoBroadcastJoinThreshold"]:
    print(f"{k:55} {spark.conf.get(k, None)}")
print("Parallélisme par défaut (cœurs) :", spark.sparkContext.defaultParallelism)
print("Version de Spark                 :", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 20:01:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/11/09 20:01:10 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
                                                                                

spark.sql.shuffle.partitions                            None
spark.sql.adaptive.enabled                              None
spark.sql.adaptive.coalescePartitions.enabled           None
spark.sql.autoBroadcastJoinThreshold                    None
Parallélisme par défaut (cœurs) : 8
Version de Spark                 : 3.5.0


In [2]:
def q_pareto():
    ventes = foi.filter("is_sale").join(dp.select("product_id", "category_en"), "product_id")
    par_cat = ventes.groupBy("category_en").agg(F.sum("price").alias("ca"))
    w = Window.orderBy(F.col("ca").desc(), "category_en")
    return par_cat.withColumn("ca_cumule", F.sum("ca").over(w.rowsBetween(Window.unboundedPreceding,
                                                                          Window.currentRow)))

def q_sessions_rfm():
    cmd = fo.filter("is_sale").select("customer_unique_id", "order_id", "order_purchase_timestamp", "items_total")
    w = Window.partitionBy("customer_unique_id").orderBy("order_purchase_timestamp", "order_id")
    s = (cmd.withColumn("prev_ts", F.lag("order_purchase_timestamp").over(w))
         .withColumn("nouvelle_session",
                     F.when(F.col("prev_ts").isNull(), 1)
                      .when(F.unix_timestamp("order_purchase_timestamp") - F.unix_timestamp("prev_ts") >= 86400, 1)
                      .otherwise(0))
         .withColumn("session_id", F.sum("nouvelle_session").over(
             w.rowsBetween(Window.unboundedPreceding, Window.currentRow))))
    return s.groupBy("customer_unique_id").agg(F.max("session_id").alias("frequence"),
                                               F.sum("items_total").alias("montant"),
                                               F.max("order_purchase_timestamp").alias("derniere"))

def q_jointures():
    return (foi.filter("is_sale")
            .join(dp.select("product_id", "category_en"), "product_id")
            .join(ds.select("seller_id", "seller_state"), "seller_id")
            .groupBy("category_en", "seller_state")
            .agg(F.sum("price").alias("ca"), F.count("*").alias("n_lignes")))

r1, r2, r3 = q_pareto(), q_sessions_rfm(), q_jointures()
print("pareto       : lignes", r1.count(), "(attendu 74), CA cumulé final :",
      f"{r1.agg(F.max('ca_cumule')).first()[0]:,.2f}", "(attendu 13 491 302,13)")
print("sessions_rfm : lignes", r2.count(), "(attendu 94961), somme des montants :",
      f"{r2.agg(F.sum('montant')).first()[0]:,.2f}", "(attendu 13 491 302,13)")
print("jointures    : groupes", r3.count(), "(mesure), somme du CA :",
      f"{r3.agg(F.sum('ca')).first()[0]:,.2f}", "(attendu 13 491 302,13)")

26/11/09 20:01:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:01:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:01:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:01:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:01:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:01:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 2

pareto       : lignes 74 (attendu 74), CA cumulé final : 13,491,302.13 (attendu 13 491 302,13)


sessions_rfm : lignes 94961 (attendu 94961), somme des montants : 13,491,302.13 (attendu 13 491 302,13)
jointures    : groupes 512 (mesure), somme du CA : 13,491,302.13 (attendu 13 491 302,13)


In [5]:
def resume_plan(plan):
    return {"shuffles": sum(plan.count(k) for k in ["Exchange hashpartitioning", "Exchange rangepartitioning",
                                                    "Exchange SinglePartition", "Exchange roundrobinpartitioning"]),
            "broadcasts": plan.count("BroadcastExchange"),
            "BroadcastHashJoin": plan.count("BroadcastHashJoin"),
            "SortMergeJoin": plan.count("SortMergeJoin"),
            "Window": plan.count("Window ["),
            "AQEShuffleRead": plan.count("AQEShuffleRead")}

def profil(nom, construire, n=5):
    sc = spark.sparkContext
    df = construire()
    df.collect()                       # échauffement ; avec AQE le plan final n'existe qu'après exécution
    plan = df._jdf.queryExecution().executedPlan().toString()
    temps = []
    for i in range(n):
        d = construire()
        sc.setJobGroup(f"{nom}_{i}", nom)
        t0 = time.perf_counter()
        d.write.format("noop").mode("overwrite").save()
        temps.append(time.perf_counter() - t0)
    suivi = sc.statusTracker()
    jobs = list(suivi.getJobIdsForGroup(f"{nom}_{n - 1}"))
    stages = set()
    for j in jobs:
        info = suivi.getJobInfo(j)
        if info:
            stages.update(info.stageIds)
    taches = sum(suivi.getStageInfo(s).numTasks for s in stages if suivi.getStageInfo(s))
    temps = sorted(temps)
    res = {"requete": nom, "mediane_s": round(temps[n // 2], 3), "min_s": round(temps[0], 3),
           "max_s": round(temps[-1], 3), "jobs": len(jobs), "stages": len(stages), "taches": taches}
    res.update(resume_plan(plan))
    return res, plan

resultats, plans = [], {}
for nom, f in [("pareto", q_pareto), ("sessions_rfm", q_sessions_rfm), ("jointures", q_jointures)]:
    r, p = profil(nom, f)
    resultats.append(r)
    plans[nom] = p

reference = pd.DataFrame(resultats)
print(reference.to_string(index=False))

26/11/09 20:02:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:02:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:02:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:02:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:02:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:02:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 2

     requete  mediane_s  min_s  max_s  jobs  stages  taches  shuffles  broadcasts  BroadcastHashJoin  SortMergeJoin  Window  AQEShuffleRead
      pareto      0.416  0.336  0.468    10      17      39         4           2                  2              0       2               1
sessions_rfm      0.427  0.343  0.437     8      11      35         2           0                  0              0       4               1
   jointures      0.453  0.370  0.561    12      17      31         2           4                  4              0       0               1


In [4]:
for nom, p in plans.items():
    print("=" * 25, nom, "=" * 25)
    print(p)

========================= pareto =========================
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   Window [sum(ca#385) windowspecdefinition(ca#385 DESC NULLS LAST, category_en#96 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS ca_cumule#389], [ca#385 DESC NULLS LAST, category_en#96 ASC NULLS FIRST]
   +- *(4) Sort [ca#385 DESC NULLS LAST, category_en#96 ASC NULLS FIRST], false, 0
      +- ShuffleQueryStage 2
         +- Exchange SinglePartition, ENSURE_REQUIREMENTS, [plan_id=1521]
            +- *(3) HashAggregate(keys=[category_en#96], functions=[sum(price#5)], output=[category_en#96, ca#385])
               +- AQEShuffleRead coalesced
                  +- ShuffleQueryStage 1
                     +- Exchange hashpartitioning(category_en#96, 200), ENSURE_REQUIREMENTS, [plan_id=1483]
                        +- *(2) HashAggregate(keys=[category_en#96], functions=[partial_sum(price#5)], output=[category_en#96, sum#394])
         

In [6]:
for k in ["spark.sql.shuffle.partitions", "spark.sql.adaptive.enabled",
          "spark.sql.adaptive.coalescePartitions.enabled", "spark.sql.autoBroadcastJoinThreshold",
          "spark.sql.adaptive.advisoryPartitionSizeInBytes"]:
    try:
        print(f"{k:55} {spark.conf.get(k)}")
    except Exception as e:
        print(f"{k:55} non lisible ({type(e).__name__})")

spark.sql.shuffle.partitions                            200
spark.sql.adaptive.enabled                              true
spark.sql.adaptive.coalescePartitions.enabled           true
spark.sql.autoBroadcastJoinThreshold                    10485760b
spark.sql.adaptive.advisoryPartitionSizeInBytes         67108864b


In [10]:
def plan_final(p):
    return p.split("== Initial Plan ==")[0]

structure = pd.DataFrame([{"requete": nom, **resume_plan(plan_final(p))} for nom, p in plans.items()])
print(structure.to_string(index=False))

suivi = spark.sparkContext.statusTracker()
for nom in ["pareto", "sessions_rfm", "jointures"]:
    print("=" * 10, nom, "(dernière exécution)")
    for j in sorted(suivi.getJobIdsForGroup(f"{nom}_4")):
        info = suivi.getJobInfo(j)
        if not info:
            continue
        det = []
        for s in info.stageIds:
            st = suivi.getStageInfo(s)
            if st:
                det.append((s, st.numTasks, st.numCompletedTasks))
        print(f"job {j} [{info.status}] stages (id, numTasks, numCompletedTasks) : {det}")

     requete  shuffles  broadcasts  BroadcastHashJoin  SortMergeJoin  Window  AQEShuffleRead
      pareto         2           1                  1              0       1               1
sessions_rfm         1           0                  0              0       2               1
   jointures         1           2                  2              0       0               1
========== pareto (dernière exécution)
job 48 [SUCCEEDED] stages (id, numTasks, numCompletedTasks) : [(81, 1, 1)]
job 49 [SUCCEEDED] stages (id, numTasks, numCompletedTasks) : [(82, 3, 3)]
job 50 [SUCCEEDED] stages (id, numTasks, numCompletedTasks) : [(84, 1, 1), (83, 3, 0)]
job 51 [SUCCEEDED] stages (id, numTasks, numCompletedTasks) : [(85, 3, 0), (86, 1, 0), (87, 1, 1)]
job 52 [SUCCEEDED] stages (id, numTasks, numCompletedTasks) : [(88, 3, 3)]
job 53 [SUCCEEDED] stages (id, numTasks, numCompletedTasks) : [(89, 3, 0), (90, 7, 7)]
job 108 [SUCCEEDED] stages (id, numTasks, numCompletedTasks) : [(171, 1, 1)]
job 109 [SUCCE

In [11]:
def mesure(nom, construire, n=5):
    construire().write.format("noop").mode("overwrite").save()      # échauffement
    t = []
    for _ in range(n):
        t0 = time.perf_counter()
        construire().write.format("noop").mode("overwrite").save()
        t.append(time.perf_counter() - t0)
    t.sort()
    return {"requete": nom, "mediane_s": round(t[n // 2], 3), "min_s": round(t[0], 3), "max_s": round(t[-1], 3)}

plancher = [
    mesure("scan 1 colonne (fact_orders)", lambda: fo.select("order_id")),
    mesure("filtre + count (fact_order_items)", lambda: foi.filter("is_sale").agg(F.count("*"))),
]
print(pd.DataFrame(plancher).to_string(index=False))

                          requete  mediane_s  min_s  max_s
     scan 1 colonne (fact_orders)      0.146  0.124  0.149
filtre + count (fact_order_items)      0.119  0.090  0.152


In [12]:
for k in ["spark.sql.adaptive.coalescePartitions.parallelismFirst",
          "spark.sql.adaptive.coalescePartitions.minPartitionSize",
          "spark.sql.adaptive.coalescePartitions.initialPartitionNum",
          "spark.sql.adaptive.skewJoin.enabled"]:
    try:
        print(f"{k:60} {spark.conf.get(k)}")
    except Exception as e:
        print(f"{k:60} non lisible ({type(e).__name__})")

spark.sql.adaptive.coalescePartitions.parallelismFirst       true
spark.sql.adaptive.coalescePartitions.minPartitionSize       1048576b
spark.sql.adaptive.coalescePartitions.initialPartitionNum    None
spark.sql.adaptive.skewJoin.enabled                          true


In [13]:
import json, urllib.request

def octets_par_stage():
    port = spark.sparkContext.uiWebUrl.rsplit(":", 1)[1]
    app = spark.sparkContext.applicationId
    url = f"http://localhost:{port}/api/v1/applications/{app}/stages?status=complete"
    with urllib.request.urlopen(url, timeout=10) as r:
        return {s["stageId"]: s for s in json.load(r)}

def profil2(nom, construire, n=15):
    sc = spark.sparkContext
    suivi = sc.statusTracker()
    tag = int(time.time())                       # noms de groupes uniques, même si la cellule est rejouée
    try:
        sc.setJobGroup(f"{nom}_{tag}_echauffement", nom)
        df = construire()
        df.collect()
        plan = df._jdf.queryExecution().executedPlan().toString()
        temps = []
        for i in range(n):
            sc.setJobGroup(f"{nom}_{tag}_run{i}", nom)
            d = construire()
            t0 = time.perf_counter()
            d.write.format("noop").mode("overwrite").save()
            temps.append(time.perf_counter() - t0)
        jobs = list(suivi.getJobIdsForGroup(f"{nom}_{tag}_run{n - 1}"))
    finally:
        sc.setJobGroup("hors_mesure", "hors mesure")   # les jobs suivants ne tombent plus dans le groupe

    executes = {}
    for j in jobs:
        info = suivi.getJobInfo(j)
        for s in (info.stageIds if info else []):
            st = suivi.getStageInfo(s)
            if st and st.numCompletedTasks > 0:
                executes[s] = st.numCompletedTasks

    ko_ecrit = ko_lu = ko_entree = None
    try:
        api = octets_par_stage()
        sel = [api[s] for s in executes if s in api]
        ko_ecrit = round(sum(s["shuffleWriteBytes"] for s in sel) / 1024, 1)
        ko_lu = round(sum(s["shuffleReadBytes"] for s in sel) / 1024, 1)
        ko_entree = round(sum(s["inputBytes"] for s in sel) / 1024, 1)
    except Exception as e:
        print(f"{nom} : API de la Spark UI indisponible ({type(e).__name__})")

    s = pd.Series(temps)
    res = {"requete": nom, "mediane_s": round(s.median(), 3), "q1_s": round(s.quantile(.25), 3),
           "q3_s": round(s.quantile(.75), 3), "min_s": round(s.min(), 3), "max_s": round(s.max(), 3),
           "jobs": len(jobs), "stages_exec": len(executes), "taches_exec": sum(executes.values()),
           "shuffle_ecrit_Ko": ko_ecrit, "shuffle_lu_Ko": ko_lu, "entree_Ko": ko_entree}
    return res, plan

cas = [("pareto", q_pareto), ("sessions_rfm", q_sessions_rfm), ("jointures", q_jointures),
       ("plancher_scan", lambda: fo.select("order_id")),
       ("plancher_count", lambda: foi.filter("is_sale").agg(F.count("*")))]

res2, plans2 = [], {}
for nom, f in cas:
    r, p = profil2(nom, f)
    res2.append(r)
    plans2[nom] = p

reference2 = pd.DataFrame(res2)
print(reference2.to_string(index=False))

26/11/09 20:14:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:14:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:14:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:14:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:14:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:14:57 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 2

       requete  mediane_s  q1_s  q3_s  min_s  max_s  jobs  stages_exec  taches_exec  shuffle_ecrit_Ko  shuffle_lu_Ko  entree_Ko
        pareto      0.414 0.384 0.435  0.362  0.546     4            4            6               8.0            8.0     3365.0
  sessions_rfm      0.626 0.604 0.702  0.474  0.899     2            2           10            7333.6         7333.6     7817.3
     jointures      0.546 0.524 0.619  0.488  0.919     4            4            6              34.1           34.1     3761.9
 plancher_scan      0.148 0.138 0.168  0.131  0.225     1            1            3               0.0            0.0     3271.8
plancher_count      0.125 0.112 0.140  0.099  0.198     2            2            4               0.2            0.2      108.4


In [14]:
CONF_BASE = {"spark.sql.autoBroadcastJoinThreshold": "10485760b",
             "spark.sql.adaptive.autoBroadcastJoinThreshold": None}
CONF_SMJ = {"spark.sql.autoBroadcastJoinThreshold": "-1",
            "spark.sql.adaptive.autoBroadcastJoinThreshold": "-1"}

def appliquer(reglages):
    for k, v in reglages.items():
        if v is None:
            spark.conf.unset(k)
        else:
            spark.conf.set(k, v)

def indicateurs(groupe):
    suivi = spark.sparkContext.statusTracker()
    jobs = list(suivi.getJobIdsForGroup(groupe))
    executes = {}
    for j in jobs:
        info = suivi.getJobInfo(j)
        for s in (info.stageIds if info else []):
            st = suivi.getStageInfo(s)
            if st and st.numCompletedTasks > 0:
                executes[s] = st.numCompletedTasks
    ecrit = None
    try:
        api = octets_par_stage()
        ecrit = round(sum(api[s]["shuffleWriteBytes"] for s in executes if s in api) / 1024, 1)
    except Exception:
        pass
    return {"jobs": len(jobs), "stages_exec": len(executes), "taches_exec": sum(executes.values()),
            "shuffle_ecrit_Ko": ecrit}

def ab(nom, construire, variantes, n=15):
    sc = spark.sparkContext
    tag = int(time.time())
    temps = {v: [] for v in variantes}
    plans_v, groupes = {}, {}
    try:
        for v, reglages in variantes.items():            # échauffement + plan final de chaque variante
            appliquer(reglages)
            sc.setJobGroup(f"{nom}_{tag}_{v}_echauffement", nom)
            df = construire()
            df.collect()
            plans_v[v] = df._jdf.queryExecution().executedPlan().toString()
        for i in range(n):                               # mesures entrelacées : A, B, A, B, ...
            for v, reglages in variantes.items():
                appliquer(reglages)
                g = f"{nom}_{tag}_{v}_run{i}"
                sc.setJobGroup(g, nom)
                d = construire()
                t0 = time.perf_counter()
                d.write.format("noop").mode("overwrite").save()
                temps[v].append(time.perf_counter() - t0)
                groupes[v] = g
    finally:
        appliquer(CONF_BASE)
        sc.setJobGroup("hors_mesure", "hors mesure")

    lignes = []
    for v in variantes:
        s = pd.Series(temps[v])
        p = plan_final(plans_v[v])
        lignes.append({"requete": nom, "variante": v, "mediane_s": round(s.median(), 3),
                       "q1_s": round(s.quantile(.25), 3), "q3_s": round(s.quantile(.75), 3),
                       "min_s": round(s.min(), 3), "max_s": round(s.max(), 3),
                       **indicateurs(groupes[v]),
                       "shuffles": resume_plan(p)["shuffles"],
                       "BHJ": p.count("BroadcastHashJoin"), "SMJ": p.count("SortMergeJoin")})
    return pd.DataFrame(lignes), plans_v

In [15]:
variantes = {"broadcast_defaut": CONF_BASE, "sortmerge_force": CONF_SMJ}

for v, reglages in variantes.items():                    # justesse avant mesure
    appliquer(reglages)
    r = q_jointures()
    print(f"{v:18} groupes : {r.count()} (attendu 512) | CA : {r.agg(F.sum('ca')).first()[0]:,.2f}")
appliquer(CONF_BASE)

tab_p, plans_p = ab("pareto", q_pareto, variantes)
tab_j, plans_j = ab("jointures", q_jointures, variantes)
ab_res = pd.concat([tab_p, tab_j], ignore_index=True)
print(ab_res.to_string(index=False))

broadcast_defaut   groupes : 512 (attendu 512) | CA : 13,491,302.13
sortmerge_force    groupes : 512 (attendu 512) | CA : 13,491,302.13


26/11/09 20:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:17:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 20:17:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/11/09 2

  requete         variante  mediane_s  q1_s  q3_s  min_s  max_s  jobs  stages_exec  taches_exec  shuffle_ecrit_Ko  shuffles  BHJ  SMJ
   pareto broadcast_defaut      0.469 0.441 0.492  0.342  0.600     4            4            6               8.0         2    1    0
   pareto  sortmerge_force      0.424 0.400 0.463  0.311  0.524     5            5            9            3187.2         4    0    1
jointures broadcast_defaut      0.577 0.505 0.628  0.464  0.775     4            4            6              34.1         1    2    0
jointures  sortmerge_force      0.736 0.687 0.862  0.642  1.035     6            6           11            5847.8         5    0    2


In [16]:
print(plan_final(plans_j["sortmerge_force"]))

AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(10) HashAggregate(keys=[category_en#96, seller_state#121], functions=[sum(price#5), count(1)], output=[category_en#96, seller_state#121, ca#7615, n_lignes#7617L])
   +- AQEShuffleRead coalesced
      +- ShuffleQueryStage 4
         +- Exchange hashpartitioning(category_en#96, seller_state#121, 200), ENSURE_REQUIREMENTS, [plan_id=29476]
            +- *(9) HashAggregate(keys=[category_en#96, seller_state#121], functions=[partial_sum(price#5), partial_count(1)], output=[category_en#96, seller_state#121, sum#7624, count#7625L])
               +- *(9) Project [price#5, category_en#96, seller_state#121]
                  +- *(9) SortMergeJoin [seller_id#3], [seller_id#118], Inner
                     :- *(7) Sort [seller_id#3 ASC NULLS FIRST], false, 0
                     :  +- AQEShuffleRead coalesced
                     :     +- ShuffleQueryStage 3
                     :        +- Exchange hashpartitioning(seller_id#3, 200), EN

In [17]:
CONF_BASE = {**CONF_BASE, "spark.sql.adaptive.enabled": "true", "spark.sql.shuffle.partitions": "200"}

var43 = {
    "aqe_on_200": CONF_BASE,
    "aqe_off_200": {**CONF_BASE, "spark.sql.adaptive.enabled": "false"},
    "aqe_off_8": {**CONF_BASE, "spark.sql.adaptive.enabled": "false", "spark.sql.shuffle.partitions": "8"},
    "aqe_on_2000": {**CONF_BASE, "spark.sql.shuffle.partitions": "2000"},
}

for v, reglages in var43.items():
    appliquer(reglages)
    r = q_sessions_rfm()
    print(f"{v:12} lignes : {r.count()} (attendu 94961) | montants : {r.agg(F.sum('montant')).first()[0]:,.2f}")
appliquer(CONF_BASE)

aqe_on_200   lignes : 94961 (attendu 94961) | montants : 13,491,302.13
aqe_off_200  lignes : 94961 (attendu 94961) | montants : 13,491,302.13
aqe_off_8    lignes : 94961 (attendu 94961) | montants : 13,491,302.13
aqe_on_2000  lignes : 94961 (attendu 94961) | montants : 13,491,302.13


In [18]:
tab_s, plans_s = ab("sessions_rfm", q_sessions_rfm, var43)
tab_j2, plans_j2 = ab("jointures", q_jointures, var43)
res43 = pd.concat([tab_s, tab_j2], ignore_index=True)
print(res43.drop(columns=["BHJ", "SMJ"]).to_string(index=False))

     requete    variante  mediane_s  q1_s  q3_s  min_s  max_s  jobs  stages_exec  taches_exec  shuffle_ecrit_Ko  shuffles
sessions_rfm  aqe_on_200      0.464 0.413 0.534  0.363  0.648     2            2           10            7333.6         1
sessions_rfm aqe_off_200      1.397 1.219 1.544  1.127  1.654     1            2          203            7333.6         1
sessions_rfm   aqe_off_8      0.440 0.417 0.487  0.388  0.538     1            2           11            7339.6         1
sessions_rfm aqe_on_2000      0.465 0.433 0.512  0.393  0.926     2            2           10            7687.1         1
   jointures  aqe_on_200      0.394 0.385 0.467  0.378  0.614     4            4            6              34.1         1
   jointures aqe_off_200      0.744 0.698 0.758  0.654  0.849     3            4          205              34.1         1
   jointures   aqe_off_8      0.498 0.472 0.544  0.439  0.646     3            4           13              16.8         1
   jointures aqe_on_2000

In [19]:
print("h")

h


In [20]:
def base_ventes():
    return foi.filter("is_sale").join(dp.select("product_id", "category_en"), "product_id")

def aggregats(base):
    return [
        base.groupBy("category_en").agg(F.sum("price").alias("ca")),
        base.groupBy("seller_id").agg(F.count("*").alias("n"), F.sum("price").alias("ca")),
        base.groupBy("category_en").agg(F.percentile_approx("price", 0.5).alias("prix_median")),
    ]

def lancer(aggs):
    for a in aggs:
        a.write.format("noop").mode("overwrite").save()

def une_iteration(avec_cache):
    t0 = time.perf_counter()
    if avec_cache:
        base = base_ventes().cache()
        base.count()                                   # matérialisation
        t_mat = time.perf_counter() - t0
        lancer(aggregats(base))
        total = time.perf_counter() - t0
        base.unpersist(blocking=True)
    else:
        lancer(aggregats(base_ventes()))
        total, t_mat = time.perf_counter() - t0, 0.0
    return total, t_mat

# vérification de justesse : mêmes résultats avec et sans cache
sans = [a.collect() for a in aggregats(base_ventes())]
b = base_ventes().cache(); b.count()
avec = [a.collect() for a in aggregats(b)]
for i, (x, y) in enumerate(zip(sans, avec), 1):
    print(f"agrégat {i} : {len(x)} lignes, identique avec cache : {sorted(map(str, x)) == sorted(map(str, y))}")

# lecture du plan et de la taille du cache
plan_c = aggregats(b)[0]._jdf.queryExecution().executedPlan().toString()
plan_s = aggregats(base_ventes())[0]._jdf.queryExecution().executedPlan().toString()
print("sans cache : FileScan =", plan_s.count("FileScan"), "| InMemoryTableScan =", plan_s.count("InMemoryTableScan"))
print("avec cache : FileScan =", plan_c.count("FileScan"), "| InMemoryTableScan =", plan_c.count("InMemoryTableScan"))
try:
    port = spark.sparkContext.uiWebUrl.rsplit(":", 1)[1]
    url = f"http://localhost:{port}/api/v1/applications/{spark.sparkContext.applicationId}/storage/rdd"
    with urllib.request.urlopen(url, timeout=10) as r:
        for rdd in json.load(r):
            print("cache :", rdd["numCachedPartitions"], "partitions | mémoire", round(rdd["memoryUsed"] / 1024, 1),
                  "Ko | disque", round(rdd["diskUsed"] / 1024, 1), "Ko")
except Exception as e:
    print("API de stockage indisponible :", type(e).__name__)
b.unpersist(blocking=True)

agrégat 1 : 74 lignes, identique avec cache : True
agrégat 2 : 3053 lignes, identique avec cache : True
agrégat 3 : 74 lignes, identique avec cache : True
sans cache : FileScan = 2 | InMemoryTableScan = 1
avec cache : FileScan = 2 | InMemoryTableScan = 1
cache : 3 partitions | mémoire 14897.0 Ko | disque 0.0 Ko


DataFrame[product_id: string, order_id: string, order_item_id: int, seller_id: string, shipping_limit_date: timestamp, price: double, freight_value: double, customer_unique_id: string, customer_state: string, order_status: string, is_sale: boolean, in_analysis_window: boolean, order_date_key: int, category_en: string]

In [21]:
une_iteration(False); une_iteration(True)               # échauffement
tot = {"sans_cache": [], "avec_cache": []}
mat = []
for i in range(15):
    t, _ = une_iteration(False); tot["sans_cache"].append(t)
    t, m = une_iteration(True);  tot["avec_cache"].append(t); mat.append(m)

lignes = []
for v, ts in tot.items():
    s = pd.Series(ts)
    lignes.append({"variante": v, "mediane_s": round(s.median(), 3), "q1_s": round(s.quantile(.25), 3),
                   "q3_s": round(s.quantile(.75), 3), "min_s": round(s.min(), 3), "max_s": round(s.max(), 3)})
print(pd.DataFrame(lignes).to_string(index=False))
print("matérialisation du cache (médiane) :", round(pd.Series(mat).median(), 3), "s")

  variante  mediane_s  q1_s  q3_s  min_s  max_s
sans_cache      1.205 1.125 1.347  1.074  1.449
avec_cache      1.631 1.578 1.788  1.514  2.237
matérialisation du cache (médiane) : 1.282 s


In [22]:
spark.catalog.clearCache()

def base_projetee():
    return base_ventes().select("category_en", "price", "seller_id")

plan_s = aggregats(base_ventes())[0]._jdf.queryExecution().executedPlan().toString()
b = base_projetee().cache()
b.count()
plan_c = aggregats(b)[0]._jdf.queryExecution().executedPlan().toString()
print("sans cache    : InMemoryTableScan =", plan_s.count("InMemoryTableScan"))
print("cache projeté : InMemoryTableScan =", plan_c.count("InMemoryTableScan"))
try:
    port = spark.sparkContext.uiWebUrl.rsplit(":", 1)[1]
    url = f"http://localhost:{port}/api/v1/applications/{spark.sparkContext.applicationId}/storage/rdd"
    with urllib.request.urlopen(url, timeout=10) as r:
        for rdd in json.load(r):
            print("cache projeté :", rdd["numCachedPartitions"], "partitions | mémoire",
                  round(rdd["memoryUsed"] / 1024, 1), "Ko | disque", round(rdd["diskUsed"] / 1024, 1), "Ko")
except Exception as e:
    print("API de stockage indisponible :", type(e).__name__)
_ = b.unpersist(blocking=True)

sans cache    : InMemoryTableScan = 0
cache projeté : InMemoryTableScan = 1
cache projeté : 3 partitions | mémoire 1969.4 Ko | disque 0.0 Ko


In [23]:
def une_iteration2(mode):
    t0 = time.perf_counter()
    if mode == "sans":
        lancer(aggregats(base_ventes()))
        return time.perf_counter() - t0, 0.0
    base = (base_ventes() if mode == "complet" else base_projetee()).cache()
    base.count()
    t_mat = time.perf_counter() - t0
    lancer(aggregats(base))
    total = time.perf_counter() - t0
    _ = base.unpersist(blocking=True)
    return total, t_mat

modes = ["sans", "complet", "projete"]
for m in modes:
    une_iteration2(m)                                   # échauffement
tot = {m: [] for m in modes}
mat = {m: [] for m in modes if m != "sans"}
for i in range(15):
    for m in modes:
        t, mt = une_iteration2(m)
        tot[m].append(t)
        if m != "sans":
            mat[m].append(mt)

lignes = []
for m, ts in tot.items():
    s = pd.Series(ts)
    lignes.append({"variante": m, "mediane_s": round(s.median(), 3), "q1_s": round(s.quantile(.25), 3),
                   "q3_s": round(s.quantile(.75), 3), "min_s": round(s.min(), 3), "max_s": round(s.max(), 3),
                   "materialisation_s": round(pd.Series(mat[m]).median(), 3) if m in mat else 0.0})
print(pd.DataFrame(lignes).to_string(index=False))

variante  mediane_s  q1_s  q3_s  min_s  max_s  materialisation_s
    sans      1.307 1.239 1.380  0.997  1.600              0.000
 complet      1.740 1.639 1.793  1.372  2.045              1.344
 projete      0.869 0.812 0.928  0.628  1.195              0.553


In [24]:
from operator import add

def paires():
    return base_ventes().select("category_en", "price").rdd.map(lambda r: (r[0], r[1]))

def v_groupbykey():
    return paires().groupByKey().mapValues(sum)

def v_reducebykey():
    return paires().reduceByKey(add)

def v_dataframe():
    return base_ventes().groupBy("category_en").agg(F.sum("price").alias("ca"))

ref = {r["category_en"]: r["ca"] for r in v_dataframe().collect()}
for nom, f in [("groupByKey", v_groupbykey), ("reduceByKey", v_reducebykey)]:
    res = dict(f().collect())
    ecart = max(abs(res[k] - ref[k]) for k in ref)
    print(f"{nom:12} clés : {len(res)} (attendu 74) | CA : {sum(res.values()):,.2f} | "
          f"écart max vs DataFrame : {ecart:.6f} | mêmes clés : {set(res) == set(ref)}")

groupByKey   clés : 74 (attendu 74) | CA : 13,491,302.13 | écart max vs DataFrame : 0.000000 | mêmes clés : True
reduceByKey  clés : 74 (attendu 74) | CA : 13,491,302.13 | écart max vs DataFrame : 0.000000 | mêmes clés : True


In [25]:
def mesure_rdd(variantes, n=15):
    sc = spark.sparkContext
    tag = int(time.time())
    temps = {v: [] for v in variantes}
    groupes = {}
    try:
        for v, f in variantes.items():                   # échauffement
            sc.setJobGroup(f"rdd_{tag}_{v}_echauffement", v)
            f().collect()
        for i in range(n):                               # mesures entrelacées
            for v, f in variantes.items():
                g = f"rdd_{tag}_{v}_run{i}"
                sc.setJobGroup(g, v)
                t0 = time.perf_counter()
                f().collect()
                temps[v].append(time.perf_counter() - t0)
                groupes[v] = g
    finally:
        sc.setJobGroup("hors_mesure", "hors mesure")
    lignes = []
    for v in variantes:
        s = pd.Series(temps[v])
        lignes.append({"variante": v, "mediane_s": round(s.median(), 3), "q1_s": round(s.quantile(.25), 3),
                       "q3_s": round(s.quantile(.75), 3), "min_s": round(s.min(), 3), "max_s": round(s.max(), 3),
                       **indicateurs(groupes[v])})
    return pd.DataFrame(lignes)

print(mesure_rdd({"rdd_groupByKey": v_groupbykey,
                  "rdd_reduceByKey": v_reducebykey,
                  "dataframe_groupBy": v_dataframe}).to_string(index=False))

         variante  mediane_s  q1_s  q3_s  min_s  max_s  jobs  stages_exec  taches_exec  shuffle_ecrit_Ko
   rdd_groupByKey      0.996 0.926 1.057  0.751  1.323     2            3            7             355.1
  rdd_reduceByKey      0.931 0.902 0.991  0.759  1.250     2            3            7               2.2
dataframe_groupBy      0.443 0.381 0.522  0.314  0.837     3            3            5               5.9


In [26]:
for nom, f in [("groupByKey", v_groupbykey), ("reduceByKey", v_reducebykey)]:
    print("=" * 10, nom)
    print(f().toDebugString().decode())

========== groupByKey
(3) PythonRDD[7946] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[7945] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[7944] at partitionBy at <unknown>:0 []
 +-(3) PairwiseRDD[7943] at groupByKey at /tmp/ipykernel_6719/1739575384.py:7 []
    |  PythonRDD[7942] at groupByKey at /tmp/ipykernel_6719/1739575384.py:7 []
    |  MapPartitionsRDD[7941] at javaToPython at <unknown>:0 []
    |  MapPartitionsRDD[7940] at javaToPython at <unknown>:0 []
    |  SQLExecutionRDD[7939] at javaToPython at <unknown>:0 []
    |  MapPartitionsRDD[7938] at javaToPython at <unknown>:0 []
    |  MapPartitionsRDD[7937] at javaToPython at <unknown>:0 []
    |  FileScanRDD[7936] at javaToPython at <unknown>:0 []
========== reduceByKey
(3) PythonRDD[7961] at RDD at PythonRDD.scala:53 []
 |  MapPartitionsRDD[7960] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[7959] at partitionBy at <unknown>:0 []
 +-(3) PairwiseRDD[7958] at reduceByKey at /tmp/ipykernel_6719/